# Error Analysis

This notebook inspects the final released taxonomy annotations and reproduces key table-style summaries.
It is intended for qualitative review and light analysis, not as the primary batch pipeline.

In [ ]:
from pathlib import Path
import pandas as pd
import sys

def find_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "src").exists() and (path / "annotations").exists():
            return path
    raise FileNotFoundError("Could not locate repository root.")

ROOT = find_root(Path.cwd())
sys.path.insert(0, str(ROOT / "src"))

from groundlm.analysis.labels import aggregated_error_rates, label_rates_by_model

path = ROOT / "annotations" / "qualitative_direct.csv"
df = pd.read_csv(path)
df.head()


In [ ]:
aggregated_error_rates(df)


In [ ]:
from groundlm.config import MODEL_ORDER, TAXONOMY_LABELS

label_long = df[["model_name", "error_label"]].copy()
label_long["label"] = (
    label_long["error_label"]
    .fillna("")
    .astype(str)
    .str.split(r"[|,;]+", regex=True)
)
label_long = label_long.explode("label")
label_long["label"] = label_long["label"].astype(str).str.strip()
label_long = label_long[label_long["label"] != ""]

counts = (
    label_long.groupby(["model_name", "label"]).size().unstack(fill_value=0)
    .reindex(index=MODEL_ORDER, columns=TAXONOMY_LABELS, fill_value=0)
)

fine_grained = counts.div(100)
fine_grained


In [ ]:
sample_index = 0
df[df["sample_index"] == sample_index][[
    "model_name",
    "error_label",
    "Review_Notes",
    "prediction",
]]
